In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


In [0]:
base_path = "/Volumes/workspace/default/etl_demo"

dbutils.fs.mkdirs(f"{base_path}/raw")
dbutils.fs.mkdirs(f"{base_path}/bronze")
dbutils.fs.mkdirs(f"{base_path}/silver")
dbutils.fs.mkdirs(f"{base_path}/gold")

True

In [0]:
files = [
    "studentInfo", "studentRegistration", "studentAssessment",
    "studentVle", "assessments", "vle", "courses"
]

In [0]:
dfs = {}
for name in files:
    dfs[name] = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{base_path}/raw/{name}.csv")
    )
    dfs[name].printSchema()

root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- region: string (nullable = true)
 |-- highest_education: string (nullable = true)
 |-- imd_band: string (nullable = true)
 |-- age_band: string (nullable = true)
 |-- num_of_prev_attempts: integer (nullable = true)
 |-- studied_credits: integer (nullable = true)
 |-- disability: string (nullable = true)
 |-- final_result: string (nullable = true)

root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- date_registration: integer (nullable = true)
 |-- date_unregistration: integer (nullable = true)

root
 |-- id_assessment: integer (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- date_submitted: integer (nullable = true)
 |-- is_banked: integer (nullable = true)
 |-- score: integer (nullable = true)

ro

In [0]:
from pyspark.sql.functions import current_timestamp

for name, df in dfs.items():
    bronze_df = df.withColumn("_ingestion_time", current_timestamp())
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{base_path}/bronze/{name}")
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_studentinfo
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/studentInfo`;

CREATE TABLE IF NOT EXISTS bronze_studentregistration
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/studentRegistration`;

CREATE TABLE IF NOT EXISTS bronze_studentassessment
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/studentAssessment`;

CREATE TABLE IF NOT EXISTS bronze_studentvle
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/studentVle`;

CREATE TABLE IF NOT EXISTS bronze_assessments
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/assessments`;

CREATE TABLE IF NOT EXISTS bronze_vle
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/vle`;

CREATE TABLE IF NOT EXISTS bronze_courses
USING DELTA AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze/courses`;

num_affected_rows,num_inserted_rows


In [0]:
from pyspark.sql.functions import col, when

# --- studentInfo ---
silver_info = (
    spark.read.format("delta").load(f"{base_path}/bronze/studentInfo")
    .dropDuplicates()
    .withColumn("studied_credits", col("studied_credits").cast("int"))
    .withColumn("num_of_prev_attempts", col("num_of_prev_attempts").cast("int"))
    .withColumn("data_quality_status",
        when(col("id_student").isNull(), "Missing ID")
        .otherwise("Valid"))
)

# --- studentRegistration ---
silver_reg = (
    spark.read.format("delta").load(f"{base_path}/bronze/studentRegistration")
    .dropDuplicates()
    .withColumn("date_registration", col("date_registration").cast("int"))
    .withColumn("date_unregistration", col("date_unregistration").cast("int"))
    .withColumn("data_quality_status",
        when(col("id_student").isNull(), "Missing ID")
        .otherwise("Valid"))
)

# --- studentAssessment ---
silver_assess = (
    spark.read.format("delta").load(f"{base_path}/bronze/studentAssessment")
    .dropDuplicates()
    .withColumn("score", col("score").cast("double"))
    .withColumn("date_submitted", col("date_submitted").cast("int"))
    .withColumn("data_quality_status",
        when(col("score").isNull(), "Missing Score")
        .otherwise("Valid"))
)

# --- studentVle ---
silver_vle = (
    spark.read.format("delta").load(f"{base_path}/bronze/studentVle")
    .dropDuplicates()
    .withColumn("date", col("date").cast("int"))
    .withColumn("sum_click", col("sum_click").cast("int"))
    .withColumn("data_quality_status",
        when(col("id_student").isNull(), "Missing ID")
        .otherwise("Valid"))
)

# --- assessments ---
silver_assessments = (
    spark.read.format("delta").load(f"{base_path}/bronze/assessments")
    .dropDuplicates()
    .withColumn("weight", col("weight").cast("double"))
    .withColumn("date", col("date").cast("int"))
    .withColumn("data_quality_status",
        when(col("id_assessment").isNull(), "Missing Assessment ID")
        .otherwise("Valid"))
)

# --- vle ---
silver_vle_meta = (
    spark.read.format("delta").load(f"{base_path}/bronze/vle")
    .dropDuplicates()
    .withColumn("week_from", col("week_from").cast("int"))
    .withColumn("week_to", col("week_to").cast("int"))
    .withColumn("data_quality_status",
        when(col("id_site").isNull(), "Missing Site ID")
        .otherwise("Valid"))
)

# --- courses ---
silver_courses = (
    spark.read.format("delta").load(f"{base_path}/bronze/courses")
    .dropDuplicates()
    .withColumn("module_presentation_length", col("module_presentation_length").cast("int"))
    .withColumn("data_quality_status",
        when(col("code_module").isNull(), "Missing Module Code")
        .otherwise("Valid"))
)

# --- Write all to silver ---
silver_tables = {
    "silver_student_info": silver_info,
    "silver_student_registration": silver_reg,
    "silver_student_assessment": silver_assess,
    "silver_student_vle": silver_vle,
    "silver_assessments": silver_assessments,
    "silver_vle_meta": silver_vle_meta,
    "silver_courses": silver_courses
}

for table_name, df in silver_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

In [0]:
%sql
-- Create views with shorter names for the silver tables
CREATE OR REPLACE VIEW silver_info AS 
SELECT * FROM silver_student_info;

CREATE OR REPLACE VIEW silver_reg AS 
SELECT * FROM silver_student_registration;

CREATE OR REPLACE VIEW silver_assess AS 
SELECT * FROM silver_student_assessment;

CREATE OR REPLACE VIEW silver_vle AS 
SELECT * FROM silver_student_vle;

CREATE OR REPLACE VIEW silver_assessments_view AS 
SELECT * FROM silver_assessments;

CREATE OR REPLACE VIEW silver_vle_meta_view AS 
SELECT * FROM silver_vle_meta;

CREATE OR REPLACE VIEW silver_courses_view AS 
SELECT * FROM silver_courses;

In [0]:
from pyspark.sql.functions import sum as _sum, count as _count, avg as _avg, col

# Read silver tables
silver_info        = spark.read.format("delta").table("silver_student_info")
silver_reg         = spark.read.format("delta").table("silver_student_registration")
silver_assess      = spark.read.format("delta").table("silver_student_assessment")
silver_vle         = spark.read.format("delta").table("silver_student_vle")
silver_assessments = spark.read.format("delta").table("silver_assessments")
silver_vle_meta    = spark.read.format("delta").table("silver_vle_meta")
silver_courses     = spark.read.format("delta").table("silver_courses")

# --- Gold 1: Average score per module & presentation ---
gold_scores_by_module = (
    silver_assess
    .filter(col("data_quality_status") == "Valid")
    .join(silver_assessments, "id_assessment")
    .groupBy("code_module", "code_presentation", "assessment_type")
    .agg(
        _avg("score").alias("AvgScore"),
        _count("*").alias("SubmissionCount"),
        _sum("score").alias("TotalScore")
    )
    .orderBy(col("AvgScore").desc())
)

# --- Gold 2: Pass/fail outcomes by region ---
gold_outcomes_by_region = (
    silver_info
    .filter(col("data_quality_status") == "Valid")
    .groupBy("region", "final_result")
    .agg(_count("*").alias("StudentCount"))
    .orderBy("region", "final_result")
)

# --- Gold 3: Pass/fail outcomes by highest education level ---
gold_outcomes_by_education = (
    silver_info
    .filter(col("data_quality_status") == "Valid")
    .groupBy("highest_education", "final_result")
    .agg(_count("*").alias("StudentCount"))
    .orderBy("highest_education", "final_result")
)

# --- Gold 4: Total clicks per module (engagement metric) ---
gold_engagement_by_module = (
    silver_vle
    .filter(col("data_quality_status") == "Valid")
    .groupBy("code_module", "code_presentation")
    .agg(
        _sum("sum_click").alias("TotalClicks"),
        _count("id_student").alias("StudentCount"),
        _avg("sum_click").alias("AvgClicksPerStudent")
    )
    .orderBy(col("TotalClicks").desc())
)

# --- Gold 5: Student engagement vs final result ---
gold_engagement_vs_outcome = (
    silver_vle
    .filter(col("data_quality_status") == "Valid")
    .groupBy("id_student", "code_module", "code_presentation")
    .agg(_sum("sum_click").alias("TotalClicks"))
    .join(
        silver_info.filter(col("data_quality_status") == "Valid")
        .select("id_student", "final_result", "region"),
        "id_student"
    )
    .groupBy("final_result", "region")
    .agg(
        _avg("TotalClicks").alias("AvgClicks"),
        _count("*").alias("StudentCount")
    )
    .orderBy("final_result", "region")
)

# --- Gold 6: Withdrawal analysis by registration data ---
gold_withdrawal_analysis = (
    silver_reg
    .filter(col("data_quality_status") == "Valid")
    .join(
        silver_info.filter(col("data_quality_status") == "Valid")
        .select("id_student", "final_result", "studied_credits"),
        "id_student"
    )
    .withColumn("withdrew",
        when(col("date_unregistration").isNotNull(), 1).otherwise(0))
    .groupBy("code_module", "code_presentation", "final_result")
    .agg(
        _count("*").alias("StudentCount"),
        _sum("withdrew").alias("WithdrawalCount"),
        _avg("studied_credits").alias("AvgStudiedCredits")
    )
    .orderBy(col("WithdrawalCount").desc())
)

# --- Write all gold tables ---
gold_tables = {
    "gold_scores_by_module":       gold_scores_by_module,
    "gold_outcomes_by_region":     gold_outcomes_by_region,
    "gold_outcomes_by_education":  gold_outcomes_by_education,
    "gold_engagement_by_module":   gold_engagement_by_module,
    "gold_engagement_vs_outcome":  gold_engagement_vs_outcome,
    "gold_withdrawal_analysis":    gold_withdrawal_analysis
}

for table_name, df in gold_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

In [0]:
%sql
SELECT * FROM bronze_studentinfo LIMIT 5;

code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,_ingestion_time
AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,2026-05-02T15:42:42.214Z
AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,2026-05-02T15:42:42.214Z
AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,2026-05-02T15:42:42.214Z
AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2026-05-02T15:42:42.214Z
AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,2026-05-02T15:42:42.214Z


In [0]:
%sql
SELECT * FROM silver_student_info LIMIT 5;

code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,_ingestion_time,data_quality_status
AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,2026-05-02T16:17:45.012Z,Valid
AAA,2013J,135335,F,East Anglian Region,Lower Than A Level,20-30%,0-35,0,180,N,Withdrawn,2026-05-02T16:17:45.012Z,Valid
AAA,2013J,151358,F,London Region,A Level or Equivalent,80-90%,0-35,0,150,N,Pass,2026-05-02T16:17:45.012Z,Valid
AAA,2013J,231554,M,London Region,A Level or Equivalent,40-50%,0-35,0,240,N,Pass,2026-05-02T16:17:45.012Z,Valid
AAA,2013J,268733,M,East Anglian Region,A Level or Equivalent,30-40%,0-35,0,180,N,Withdrawn,2026-05-02T16:17:45.012Z,Valid


In [0]:
%sql
SELECT * FROM gold_scores_by_module ORDER BY AvgScore DESC;

code_module,code_presentation,assessment_type,AvgScore,SubmissionCount,TotalScore
BBB,2013B,CMA,88.34620716973659,5049,446060.0
BBB,2013J,CMA,88.07980049875312,6416,565120.0
BBB,2014B,CMA,87.67415980413978,4493,393920.0
GGG,2013J,CMA,87.1869831955188,3749,326864.0
GGG,2014J,CMA,85.86967601019293,2747,235884.0
GGG,2014B,CMA,85.74600065295462,3063,262640.0
EEE,2014J,TMA,82.46234893089557,3227,266106.0
EEE,2013J,TMA,80.62894828184658,2881,232292.0
FFF,2014J,CMA,80.61895681435783,8915,718718.0
FFF,2013J,CMA,79.92132926415734,8847,707064.0
